# Notebook 07: SHAP Interpretability

**Project:** Cancer Microbiome Comparative Analysis  
**Description:** This notebook uses SHAP (SHapley Additive exPlanations) to interpret the best classifier trained in Notebook 06, identify which microbial genera most influence each cancer-type prediction, and cross-validate those findings against differential abundance results.

**Inputs (from Results/):**
- `abund_combined_clr.csv` — CLR-transformed genus-level abundance
- `meta_combined.csv` — metadata
- `best_model_multiclass.pkl` — trained best classifier
- `rf_feature_importances.csv` — RF importances (fallback if SHAP unavailable)
- `diff_abundance_colorectal.csv` — differential abundance results (Colorectal)
- `diff_abundance_breast.csv` — differential abundance results (Breast)

**Outputs:**
- `Figures/fig11_shap_beeswarm.png`
- `Figures/fig12_shap_bar.png`
- `Figures/fig11b_shap_da_scatter.png`
- `Results/shap_feature_importance.csv`
- `Results/shap_per_class_top_genera.csv`
- `Results/shap_da_overlap.csv`

In [ ]:
# Cell 1 — Imports
import pandas as pd     # pandas: tables and data manipulation (like Excel in Python)
import numpy as np      # numpy: fast math on arrays of numbers
import matplotlib       # matplotlib: core Python charting library
matplotlib.use('Agg')   # 'Agg' = save plots to files instead of opening pop-up windows
import matplotlib.pyplot as plt  # pyplot: easier interface for drawing charts
import seaborn as sns   # seaborn: prettier statistical charts built on matplotlib
import warnings         # warnings: controls Python's warning messages
warnings.filterwarnings('ignore')  # suppress harmless warnings to keep output clean
import os               # os: file and folder path utilities
import joblib           # joblib: loads saved machine learning models from disk (.pkl files)

# Try to import SHAP — a library that explains AI model predictions
# SHAP = SHapley Additive exPlanations, named after Shapley values from game theory
# It answers: "For this prediction, how much did each bacterium contribute?"
try:
    import shap
    shap_available = True
    print(f'SHAP version: {shap.__version__}')
except ImportError:
    print('SHAP not installed. pip install shap')
    print('Falling back to RF feature importances for all figures.')
    shap_available = False  # flag used in later cells to skip SHAP if missing

print('All core imports successful.')

In [ ]:
# Cell 2 — Paths and Data Loading
import re  # re: regular expressions library — used to clean up special characters in column names

# Build the base directory by going up one folder from wherever this notebook lives
# os.getcwd() = current working directory (the Notebooks/ folder)
# os.path.join(..., '..') = "go up one level to the parent folder"
# os.path.abspath() = convert to a full absolute path with no '..' in it
BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')     # where saved tables and models live
FIGURES_DIR = os.path.join(BASE_DIR, 'Figures')     # where saved chart images live

os.makedirs(RESULTS_DIR, exist_ok=True)  # create folder if it doesn't already exist
os.makedirs(FIGURES_DIR, exist_ok=True)  # exist_ok=True means no error if already there

print(f'BASE_DIR    : {BASE_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'FIGURES_DIR : {FIGURES_DIR}')

# Load the CLR-transformed abundance table saved by Notebook 02
# CLR = Centered Log-Ratio transformation: turns fractions (0–1) into normal-ish numbers
# Each row = one sample, each column = one bacterial genus
abund_clr = pd.read_csv(os.path.join(RESULTS_DIR, 'abund_combined_clr.csv'), index_col=0)

# Load metadata: sample IDs mapped to cancer_type and condition (Cancer vs Healthy)
meta = pd.read_csv(os.path.join(RESULTS_DIR, 'meta_combined.csv'), index_col=0)

# Keep only samples that appear in BOTH tables (inner-join style alignment)
# Important: if a sample is in one file but not the other, we can't use it
common_idx = abund_clr.index.intersection(meta.index)
abund_clr  = abund_clr.loc[common_idx]
meta       = meta.loc[common_idx]

# XGBoost (used in Notebook 06) cannot handle column names containing [ ] or <
# re.sub(pattern, replacement, string) replaces any of those characters with '_'
# We MUST apply the same renaming here so that column names match the saved model
abund_clr.columns = [re.sub(r'[\[\]<>]', '_', col) for col in abund_clr.columns]

print(f'\nCLR abundance  : {abund_clr.shape}')  # (samples, genera)
print(f'Metadata       : {meta.shape}')

# Keep only cancer patient samples — we trained the classifier on cancer only
# This matches exactly what Notebook 06 did for its Task 1 (multi-class cancer type)
cancer_mask = meta['condition'] == 'Cancer'
X_cancer    = abund_clr.loc[cancer_mask].copy()   # features for cancer samples
meta_cancer = meta.loc[cancer_mask].copy()        # metadata for cancer samples

print(f'\nCancer-only samples : {X_cancer.shape[0]}')
print(f'Class distribution  :')
print(meta_cancer['cancer_type'].value_counts())

# LabelEncoder: converts text labels into numbers the model can use
# Breast → 0, Colorectal → 1, Prostate → 2 (alphabetical order)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_cancer = le.fit_transform(meta_cancer['cancer_type'])  # array of 0s, 1s, 2s

# class_names_ordered[i] = the cancer name for class number i
# Used in later cells to label plots with real names instead of numbers
class_names_ordered = le.classes_.tolist()

print(f'\nLabel encoding:')
for i, cls in enumerate(class_names_ordered):
    print(f'  {i} -> {cls}')

# Reproduce the EXACT same 80/20 train/test split as Notebook 06
# random_state=42 ensures the same split every run (reproducibility)
# stratify=y_cancer ensures each split has roughly equal cancer-type proportions
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.20, random_state=42, stratify=y_cancer
)
print(f'\nX_test shape : {X_test.shape}')  # (70, 259) — 70 samples, 259 genera

# Load the best ML model saved by Notebook 06 (.pkl = "pickle" binary format)
# Pipeline wraps preprocessing (StandardScaler) + classifier (SVM) in one object
model_path = os.path.join(RESULTS_DIR, 'best_model_multiclass.pkl')
if os.path.exists(model_path):
    best_model = joblib.load(model_path)  # joblib.load reads the binary file back into Python
    print(f'\nLoaded best model from: {model_path}')
    print(f'Model type: {type(best_model).__name__}')
else:
    print(f'WARNING: Model file not found at {model_path}.')
    print('Please run Notebook 06 first to generate the model.')
    best_model = None

# Load Random Forest feature importances from NB06 as a fallback
# Used in figures when SHAP is unavailable (e.g., SHAP fails on SVM)
fi_path = os.path.join(RESULTS_DIR, 'rf_feature_importances.csv')
if os.path.exists(fi_path):
    fi_df = pd.read_csv(fi_path)
    print(f'\nLoaded RF feature importances: {fi_df.shape}')
else:
    fi_df = None
    print(f'RF feature importances not found — run Notebook 06 first.')

In [ ]:
# Cell 3 — SHAP Value Computation
# SHAP = SHapley Additive exPlanations (from cooperative game theory)
# For each prediction, SHAP assigns each feature (bacterium) a "credit" score:
#   positive SHAP → this bacterium pushed the prediction TOWARD this cancer class
#   negative SHAP → this bacterium pushed the prediction AWAY from this cancer class

shap_values  = None   # will hold the computed SHAP arrays
explainer    = None   # will hold the SHAP explainer object
feature_names = X_test.columns.tolist()  # list of 259 genus names — used for plot labels

# Helper: extract the final estimator from a sklearn Pipeline
# A Pipeline is a chain of steps (e.g., StandardScaler → SVM); the last step is the model
def extract_leaf_model(pipeline_or_model):
    from sklearn.pipeline import Pipeline
    if isinstance(pipeline_or_model, Pipeline):
        return pipeline_or_model.steps[-1][1]   # [-1][1] = last step's model object
    return pipeline_or_model  # if it's already a plain model, return as-is

# Helper: apply all preprocessing steps (e.g., StandardScaler) but skip the final estimator
# SHAP needs the data in the same format the model's last layer sees — after scaling
def transform_through_pipeline(pipeline_or_model, X):
    from sklearn.pipeline import Pipeline
    if isinstance(pipeline_or_model, Pipeline) and len(pipeline_or_model.steps) > 1:
        X_transformed = X
        for _, step_obj in pipeline_or_model.steps[:-1]:  # all steps except the last
            X_transformed = step_obj.transform(X_transformed)
        return X_transformed
    return X.values if hasattr(X, 'values') else X  # if no pipeline, just return the raw array

# Helper: squeeze a SHAP array into exactly 2D shape (n_samples, n_features)
# SHAP can return arrays with extra dimensions; this normalizes them
def _to_2d(sv):
    arr = np.array(sv)
    while arr.ndim > 2:
        if arr.shape[-1] == 1:
            arr = arr[..., 0]    # drop a trailing dimension of size 1
        elif arr.shape[0] == 1:
            arr = arr[0]         # drop a leading dimension of size 1
        else:
            arr = arr.reshape(arr.shape[0], -1)  # flatten all extra dimensions
            break
    return arr

# Helper: normalize raw SHAP output into a list of 2D arrays, one per class
# Handles both (n_samples, n_features, n_classes) and list formats
def _normalize_shap_list(sv_raw):
    if isinstance(sv_raw, list):
        return [_to_2d(sv) for sv in sv_raw]  # each element in the list → 2D
    arr = np.array(sv_raw)
    if arr.ndim == 3:
        # 3D array: last axis = class; split into a list along that axis
        return [arr[:, :, i] for i in range(arr.shape[2])]
    return [_to_2d(arr)]

if shap_available and best_model is not None:
    print('Computing SHAP values with TreeExplainer...')

    leaf_clf = extract_leaf_model(best_model)  # get the bare SVM (or RF, etc.)

    # Apply the pipeline's preprocessing so data is in the right format for the model
    X_test_transformed = transform_through_pipeline(best_model, X_test)

    # Convert to a plain NumPy array — SHAP requires raw arrays, not DataFrames
    X_test_arr = (X_test_transformed.values
                  if hasattr(X_test_transformed, 'values')
                  else np.array(X_test_transformed))

    try:
        # TreeExplainer is the fastest SHAP method — works for tree-based models (RF, XGBoost)
        # It will FAIL here because the best model is SVM (not a tree)
        explainer   = shap.TreeExplainer(leaf_clf)
        shap_raw    = explainer.shap_values(X_test_arr)
        shap_values = _normalize_shap_list(shap_raw)

        print(f'SHAP values: list of {len(shap_values)} arrays')
        for i, sv in enumerate(shap_values):
            print(f'  class {i} ({class_names_ordered[i]}): {sv.shape}')
        print('SHAP computation complete.')

    except Exception as e:
        # TreeExplainer failed because SVM is not a tree model
        # Fall back to KernelExplainer — works for ANY model but is much slower
        # KernelExplainer treats the model as a black box and estimates SHAP numerically
        print(f'TreeExplainer failed: {e}')
        print('Attempting KernelExplainer (slower fallback)...')
        try:
            # Use a small background dataset to estimate baseline predictions
            # shap.sample() randomly picks a subset (up to 50 samples) for speed
            background  = shap.sample(X_test_arr, min(50, X_test_arr.shape[0]))

            # KernelExplainer uses predict_proba (probability output) for multi-class SVM
            explainer   = shap.KernelExplainer(best_model.predict_proba, background)

            # Only compute on first 20 test samples — KernelExplainer is SLOW (minutes per sample)
            # This creates a shape mismatch: SHAP will have 20 rows, X_test has 70 rows
            shap_raw    = explainer.shap_values(X_test_arr[:20])
            shap_values = _normalize_shap_list(shap_raw)
            print('KernelExplainer complete (on first 20 test samples).')
            # WARNING: 20 samples vs X_test's 70 rows → beeswarm plots will be skipped in Cell 4
        except Exception as e2:
            print(f'KernelExplainer also failed: {e2}')
            shap_values = None
else:
    X_test_arr = X_test.values  # still need this array for the scatter plot in Cell 8
    print('SHAP not available or model not loaded — skipping computation.')

In [ ]:
# Cell 4 — Figure 11: SHAP Beeswarm Summary Plots
# A beeswarm plot shows every single sample as a dot:
#   - X position: SHAP value (positive → pushes toward this cancer class)
#   - Y position: which bacterium (feature)
#   - Color: actual abundance of that bacterium (red = high, blue = low)
# Together this tells us: "Which bacteria drive predictions, and in which direction?"
#
# WHY we save separate figures and then combine them:
#   shap.summary_plot() creates its own matplotlib figure internally
#   It cannot be placed inside a pre-made subplot grid (plt.sca(ax) doesn't redirect it)
#   Trying to do so causes a shape mismatch error in SHAP's internal code
# Strategy: save each class as its own PNG, then stitch them together with matplotlib

if shap_available and shap_values is not None:
    print('Generating SHAP beeswarm plots (Figure 11) ...')

    # Normalize shap_values to a list of 2D arrays, one per cancer class
    if isinstance(shap_values, list):
        sv_list = []
        for sv in shap_values:
            sv_arr = np.array(sv)
            if sv_arr.ndim == 3:               # (n_samples, n_features, 1) → squeeze last axis
                sv_arr = sv_arr.squeeze(-1)
            sv_list.append(sv_arr)
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        # (n_samples, n_features, n_classes) format — split along last axis
        sv_list = [shap_values[:, :, i] for i in range(shap_values.shape[2])]
    else:
        sv_list = [np.array(shap_values)]

    n_cls = min(len(sv_list), len(class_names_ordered))
    print(f'  {n_cls} class(es), each SHAP array shape: {sv_list[0].shape}')

    n_feat_rows = X_test_arr.shape[0]   # how many rows in our feature matrix (70 samples)
    per_class_paths = []

    for i in range(n_cls):
        cls_name = class_names_ordered[i]
        sv_i     = sv_list[i]  # SHAP values for class i; shape: (n_shap_samples, n_features)

        # Guard: SHAP row count must match X_test_arr row count for beeswarm to work
        # KernelExplainer was run on only 20 samples, but X_test has 70 → mismatch → SKIP
        if sv_i.shape[0] != n_feat_rows:
            if sv_i.T.shape[0] == n_feat_rows:
                sv_i = sv_i.T  # try transposing as a fix
                print(f'  [{cls_name}] transposed SHAP array to match X_test rows')
            else:
                print(f'  [{cls_name}] SKIP — cannot align SHAP rows '
                      f'({sv_i.shape[0]}) with X_test rows ({n_feat_rows})')
                continue  # skip this class; beeswarm requires identical sample counts

        # Create a fresh figure for each class (SHAP takes it over internally)
        fig_i = plt.figure(figsize=(9, 7))
        try:
            shap.summary_plot(
                sv_i,                         # SHAP values: (n_samples, n_features)
                X_test_arr,                   # actual feature values for color-coding
                feature_names=feature_names,  # genus names for y-axis labels
                plot_type='dot',              # 'dot' = beeswarm; 'bar' = bar chart
                max_display=15,               # show only top 15 most important genera
                show=False                    # don't pop up a window; we'll save manually
            )
            plt.title(f'SHAP Beeswarm — {cls_name}', fontsize=12, fontweight='bold', pad=8)
        except Exception as e_inner:
            # If beeswarm still fails, fall back to a simple bar chart of mean |SHAP|
            print(f'  [{cls_name}] beeswarm failed ({e_inner}); using bar fallback')
            plt.clf()  # clear the figure
            mean_abs = np.abs(sv_i).mean(axis=0)    # average absolute SHAP per genus
            top_idx  = np.argsort(mean_abs)[-15:][::-1]  # indices of top 15
            plt.barh([feature_names[j] for j in reversed(top_idx)],
                     mean_abs[top_idx[::-1]], color='#FF6B6B')
            plt.xlabel('Mean |SHAP Value|', fontsize=11)
            plt.title(f'SHAP (bar fallback) — {cls_name}', fontsize=12, fontweight='bold')

        plt.tight_layout()
        p = os.path.join(FIGURES_DIR, f'fig11_shap_{cls_name.lower()}.png')
        plt.savefig(p, dpi=300, bbox_inches='tight')
        plt.close()  # free memory
        per_class_paths.append(p)
        print(f'  Saved: {os.path.basename(p)}')

    # Stitch individual class PNGs into one composite image for the paper
    if per_class_paths:
        from matplotlib.image import imread as mpl_imread
        imgs = [mpl_imread(p) for p in per_class_paths]  # load each PNG as a pixel array
        fig, axes = plt.subplots(1, len(imgs), figsize=(11 * len(imgs), 9))
        if len(imgs) == 1:
            axes = [axes]  # ensure axes is always a list
        for ax, img in zip(axes, imgs):
            ax.imshow(img)   # display the PNG as an image inside a matplotlib panel
            ax.axis('off')   # hide x/y tick marks — this is a photo, not a chart
        plt.tight_layout(pad=0.3)
        fig_path = os.path.join(FIGURES_DIR, 'fig11_shap_beeswarm.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'\nComposite saved: {fig_path}')

else:
    # Fallback when SHAP is unavailable: show RF importance as Figure 11
    print('SHAP unavailable — generating fallback RF importance plot (Figure 11).')
    if fi_df is not None:
        top20_fi = fi_df.head(20).copy()
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(top20_fi['genus'][::-1], top20_fi['importance'][::-1],
                color='#1976D2', edgecolor='white')
        ax.set_xlabel('RF Feature Importance', fontsize=12)
        ax.set_title('Top 20 Genera by RF Importance\n(SHAP not available)',
                     fontsize=13, fontweight='bold')
        sns.despine(ax=ax)
        plt.tight_layout()
        fig_path = os.path.join(FIGURES_DIR, 'fig11_shap_beeswarm.png')
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Saved (fallback): {fig_path}')
    else:
        print('No feature importances available. Skipping Figure 11.')

In [ ]:
# Cell 5 — Figure 12: Global SHAP Bar Chart
# This figure summarizes the MOST IMPORTANT bacteria across ALL cancer classes
# For each bacterium, we compute: mean(|SHAP value|) averaged over all samples and all classes
# Think of it as asking: "On average, how much does this bacterium affect the model's decisions?"
# The bacterium with the highest mean |SHAP| is the model's most important feature overall

shap_df_global = None  # will hold a DataFrame with genus names and their SHAP importance scores

if shap_available and shap_values is not None:
    print('Computing mean |SHAP| per feature (Figure 12)...')

    if isinstance(shap_values, list):
        # shap_values is a list of (n_samples, n_features) arrays — one per class
        # np.abs() takes absolute value (SHAP can be negative → we want magnitude only)
        # .mean(axis=0) averages across samples → shape becomes (n_features,)
        mean_abs_per_class = [np.abs(sv).mean(axis=0) for sv in shap_values]

        # Now average across classes → one importance score per bacterium
        # "mean across classes" treats all cancer types equally
        mean_shap = np.mean(mean_abs_per_class, axis=0)  # shape: (n_features,)
    else:
        mean_shap = np.abs(shap_values).mean(axis=0)  # if not a list, just average directly

    # Build a tidy DataFrame: genus name paired with its global SHAP importance
    # sort_values puts the most important bacteria at the top
    shap_df_global = pd.DataFrame({
        'genus'        : feature_names,  # list of 259 bacterial genus names
        'mean_abs_shap': mean_shap       # one float per genus
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    top20_shap = shap_df_global.head(20)  # take the top 20 most important

    # Draw horizontal bar chart (barh) — longer bar = more important bacterium
    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(
        top20_shap['genus'][::-1],         # reversed so most important is at TOP
        top20_shap['mean_abs_shap'][::-1], # corresponding bar lengths
        color='#FF6B6B',                   # salmon-red color
        edgecolor='white'                  # white edge between bars for readability
    )
    ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
    ax.set_title('Top 20 Most Important Microbial Features (SHAP)', fontsize=14, fontweight='bold')
    sns.despine(ax=ax)  # remove top and right border lines for a cleaner look
    plt.tight_layout()

    fig_path = os.path.join(FIGURES_DIR, 'fig12_shap_bar.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved: {fig_path}')

    # Save full SHAP importance table (all 259 genera, not just top 20)
    shap_csv = os.path.join(RESULTS_DIR, 'shap_feature_importance.csv')
    shap_df_global.to_csv(shap_csv, index=False)
    print(f'Saved: {shap_csv}')

    print(f'\nTop 10 by mean |SHAP|:')
    print(shap_df_global.head(10).to_string(index=False))  # print without row numbers

else:
    # Fallback: use RF feature importances instead of SHAP scores
    # RF importance = how much each feature reduces prediction error across all trees
    print('SHAP unavailable — generating fallback bar chart from RF importances (Figure 12).')

    if fi_df is not None:
        top20_fi = fi_df.head(20).copy()

        # Rename column so downstream code sees a consistent column name
        shap_df_global = fi_df.rename(columns={'importance': 'mean_abs_shap'})[['genus', 'mean_abs_shap']].copy()

        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(top20_fi['genus'][::-1], top20_fi['importance'][::-1],
                color='#FF6B6B', edgecolor='white')
        ax.set_xlabel('RF Feature Importance (proxy for Mean |SHAP|)', fontsize=12)
        ax.set_title('Top 20 Most Important Microbial Features\n(RF Importance — SHAP not available)',
                     fontsize=13, fontweight='bold')
        sns.despine(ax=ax)
        plt.tight_layout()

        fig_path = os.path.join(FIGURES_DIR, 'fig12_shap_bar.png')
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Saved (fallback): {fig_path}')

        shap_csv = os.path.join(RESULTS_DIR, 'shap_feature_importance.csv')
        shap_df_global.to_csv(shap_csv, index=False)
        print(f'Saved (fallback): {shap_csv}')
    else:
        print('No feature importances available. Skipping Figure 12.')

In [ ]:
# Cell 6 — Per-Class SHAP Analysis
# Goal: for each cancer type, find the bacteria that PUSH the model most strongly
# toward predicting that type.
# We focus on POSITIVE SHAP values only:
#   positive = "this bacterium's abundance increases the probability of this cancer class"
# .clip(min=0) sets all negative values to 0 before averaging — like zeroing out negatives
# .mean(axis=0) averages across samples → one score per genus

per_class_records = []  # will collect rows for the output CSV

if shap_available and shap_values is not None and isinstance(shap_values, list):
    print('=== Per-class SHAP Analysis ===')
    print('Top 10 genera with highest mean positive SHAP per class\n')

    for i, cls_name in enumerate(class_names_ordered):
        sv_i = shap_values[i]  # SHAP array for class i; shape: (n_shap_samples, n_features)

        # clip(min=0): replace negative SHAP values with 0 (keep only positive influence)
        # mean(axis=0): average the positive-only SHAP over all samples
        # Result: one "mean positive SHAP" score per genus
        mean_pos_shap = sv_i.clip(min=0).mean(axis=0)

        # Build a DataFrame for this class and sort by score (highest first)
        cls_df = pd.DataFrame({
            'genus'        : feature_names,      # the 259 genus names
            'mean_pos_shap': mean_pos_shap        # how much each genus pushes toward this class
        }).sort_values('mean_pos_shap', ascending=False).head(10)  # top 10 only

        print(f'--- {cls_name} ---')
        for rank_i, (_, row) in enumerate(cls_df.iterrows(), 1):
            print(f'  {rank_i:2d}. {row["genus"]:40s}  mean_pos_SHAP={row["mean_pos_shap"]:.6f}')
            per_class_records.append({
                'cancer_class' : cls_name,
                'rank'         : rank_i,
                'genus'        : row['genus'],
                'mean_pos_shap': round(row['mean_pos_shap'], 6)
            })
        print()

elif fi_df is not None:
    # Fallback: RF importance is not class-specific, but we apply the same list to each class
    # NOTE: RF importance measures overall importance across all splits in all trees
    # It does NOT tell us which class benefits — that's a limitation of this fallback
    print('SHAP not available — using RF global importance as fallback for per-class analysis.')
    print('NOTE: RF importances are not class-specific; same ranking applied to all classes.\n')

    top10_fi = fi_df.head(10)
    for cls_name in class_names_ordered:
        print(f'--- {cls_name} (RF importance, class-agnostic) ---')
        for rank_i, (_, row) in enumerate(top10_fi.iterrows(), 1):
            print(f'  {rank_i:2d}. {row["genus"]:40s}  importance={row["importance"]:.6f}')
            per_class_records.append({
                'cancer_class' : cls_name,
                'rank'         : rank_i,
                'genus'        : row['genus'],
                'mean_pos_shap': round(row['importance'], 6)
            })
        print()

else:
    print('No SHAP values or RF importances available. Skipping per-class analysis.')

# Save per-class results as a CSV table
if per_class_records:
    per_class_df = pd.DataFrame(per_class_records)
    per_class_csv = os.path.join(RESULTS_DIR, 'shap_per_class_top_genera.csv')
    per_class_df.to_csv(per_class_csv, index=False)  # index=False: don't write row numbers
    print(f'Saved: {per_class_csv}')
else:
    per_class_df = pd.DataFrame()  # empty DataFrame so Cell 7 doesn't crash
    print('No per-class records to save.')

In [ ]:
# Cell 7 — SHAP × Differential Abundance Cross-Reference
# Goal: compare two independent methods that find "important" bacteria
#   1. SHAP importance: bacteria that help the ML model distinguish cancer types
#   2. Differential abundance (DA): bacteria statistically enriched in cancer vs healthy
# If a bacterium shows up in BOTH lists, it's a more reliable biomarker candidate
# (Two different methods agree → less likely to be a false positive)

# File paths for DA results produced by Notebook 05
DA_FILES = {
    'Colorectal': os.path.join(RESULTS_DIR, 'diff_abundance_colorectal.csv'),
    'Breast'    : os.path.join(RESULTS_DIR, 'diff_abundance_breast.csv'),
}

# Load DA results — if a file is missing, skip that cancer type gracefully
da_data = {}  # dict: cancer_type → DataFrame
for cancer_type, da_path in DA_FILES.items():
    if os.path.exists(da_path):
        try:
            da_df = pd.read_csv(da_path, index_col=0)   # genus names are the row index
            da_data[cancer_type] = da_df
            print(f'Loaded DA results for {cancer_type}: {da_df.shape}')
            print(f'  Columns: {list(da_df.columns)}')
        except Exception as e:
            print(f'Could not load {da_path}: {e}')
    else:
        print(f'DA file not found: {da_path}')

# Helper function: find a column by trying multiple possible names
# Different notebooks may use different naming conventions (e.g., 'adj_p' vs 'padj')
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c   # return the first matching column name found
    return None  # none of the candidates exist in this DataFrame

# Lists of possible column names for p-values and fold changes
# We check these in priority order (most descriptive first)
P_CANDIDATES  = ['adj_p', 'padj', 'adjusted_pvalue', 'p_adj', 'q_value', 'FDR',
                 'pvalue', 'p_value', 'p']
FC_CANDIDATES = ['log2fc', 'log2FoldChange', 'log2_fold_change', 'logFC', 'log2FC',
                 'fc', 'fold_change']

overlap_records = []  # collect rows for the output CSV

if per_class_df.empty:
    print('\nNo per-class SHAP data available — skipping overlap analysis.')
else:
    print('\n=== SHAP × Differential Abundance Overlap ===')

    for cancer_type in ['Colorectal', 'Breast']:
        if cancer_type not in da_data:
            print(f'\n{cancer_type}: DA data not available — skipping.')
            continue

        da_df = da_data[cancer_type].copy()

        # Identify which columns hold p-values and fold changes in this DA table
        p_col  = find_col(da_df, P_CANDIDATES)
        fc_col = find_col(da_df, FC_CANDIDATES)

        if p_col is None:
            print(f'{cancer_type}: Cannot find p-value column. Columns: {list(da_df.columns)}')
            continue

        # "Enriched" = statistically significant (adj_p < 0.05) AND higher in cancer than healthy
        sig_mask     = da_df[p_col] < 0.05                    # True where adj_p < threshold
        enriched_mask = sig_mask & (da_df[fc_col] > 0) if fc_col else sig_mask  # AND positive FC

        # Extract the set of enriched genus names
        # The genus names might be in a 'genus' column OR they might be the row index
        if 'genus' in da_df.columns:
            enriched_genera = set(da_df.loc[enriched_mask, 'genus'].astype(str))
        else:
            enriched_genera = set(da_df.index[enriched_mask].astype(str))

        print(f'\n{cancer_type}:')
        print(f'  Enriched genera (adj_p < 0.05, FC > 0) : {len(enriched_genera)}')

        # Top 10 SHAP genera for this cancer class (from Cell 6)
        shap_top_genera = set(
            per_class_df.loc[per_class_df['cancer_class'] == cancer_type, 'genus'].astype(str)
        )
        print(f'  SHAP top genera (top 10)               : {len(shap_top_genera)}')

        # Set intersection (∩): genera appearing in BOTH lists
        overlap = shap_top_genera & enriched_genera
        print(f'  Overlap (SHAP ∩ DA enriched)           : {len(overlap)}')

        if overlap:
            print(f'  Overlapping genera:')
            for g in sorted(overlap):
                print(f'    - {g}')
        else:
            print(f'  No overlap found.')  # 0 overlap is a valid scientific result

        # Build overlap records: for every SHAP top-10 genus, record its DA stats
        for g in shap_top_genera:
            is_enriched = g in enriched_genera  # True/False: is this genus also DA-enriched?

            # Look up this genus's SHAP rank
            shap_rank_arr = per_class_df.loc[
                (per_class_df['cancer_class'] == cancer_type) &
                (per_class_df['genus'] == g), 'rank'
            ].values
            shap_rank = int(shap_rank_arr[0]) if len(shap_rank_arr) > 0 else None

            # Look up this genus's DA fold change and adjusted p-value
            if 'genus' in da_df.columns:
                da_row = da_df.loc[da_df['genus'] == g]   # match by column value
            else:
                da_row = da_df.loc[da_df.index == g]       # match by row index

            da_fc   = float(da_row[fc_col].values[0])  if (fc_col and not da_row.empty) else np.nan
            da_padj = float(da_row[p_col].values[0])   if not da_row.empty else np.nan

            overlap_records.append({
                'cancer_type': cancer_type,
                'genus'      : g,
                'shap_rank'  : shap_rank,
                'da_enriched': is_enriched,
                'da_log2fc'  : round(da_fc,   4) if not np.isnan(da_fc)   else np.nan,
                'da_adj_p'   : round(da_padj, 6) if not np.isnan(da_padj) else np.nan,
                'in_overlap' : g in overlap
            })

# Save the full overlap table (all SHAP top-10 genera with their DA stats)
if overlap_records:
    overlap_df = pd.DataFrame(overlap_records)
    overlap_csv = os.path.join(RESULTS_DIR, 'shap_da_overlap.csv')
    overlap_df.to_csv(overlap_csv, index=False)
    print(f'\nSaved overlap table: {overlap_csv}')
    # Show only rows where both methods agree (in_overlap = True)
    print(f'\nOverlap table preview:')
    print(overlap_df[overlap_df['in_overlap']].to_string(index=False))
else:
    overlap_df = pd.DataFrame()  # empty DataFrame so Cell 8 doesn't crash
    print('No overlap records generated.')

In [ ]:
# Cell 8 — Figure 11b: SHAP vs Differential Abundance Validation Scatter Plot
# Scatter plot comparing two independent measures for each bacterium:
#   X-axis: log2 fold change (from DA in NB05) — positive = more abundant in cancer than healthy
#   Y-axis: SHAP importance — how much this bacterium helps the ML model
#   Color: red = statistically significant (adj_p < 0.05), grey = not significant
#   Annotated: top-right corner = enriched AND important (the most interesting biomarkers)
#
# WHY: if a bacterium is enriched in cancer AND SHAP-important, two methods agree → more trustworthy

SCATTER_TYPES = ['Colorectal', 'Breast']  # cancer types with DA data

# Count how many cancer types actually have DA data loaded
n_scatter = sum(ct in da_data for ct in SCATTER_TYPES)

if n_scatter == 0 or overlap_df.empty:
    print('DA data or overlap data not available — skipping scatter plot (Figure 11b).')
else:
    valid_types = [ct for ct in SCATTER_TYPES if ct in da_data]

    # Create one subplot panel per cancer type (side by side)
    fig, axes = plt.subplots(1, len(valid_types), figsize=(9 * len(valid_types), 7))
    if len(valid_types) == 1:
        axes = [axes]  # ensure axes is always a list, even with one panel

    for ax, cancer_type in zip(axes, valid_types):
        da_df = da_data[cancer_type].copy()

        p_col  = find_col(da_df, P_CANDIDATES)   # find the p-value column
        fc_col = find_col(da_df, FC_CANDIDATES)  # find the fold-change column

        if p_col is None or fc_col is None:
            ax.set_title(f'{cancer_type}\n(Missing DA columns)')
            continue  # skip this panel if required columns are absent

        # Build a lookup dictionary: genus name → SHAP importance score
        # Use global SHAP importance (Cell 5) if available; otherwise use RF importance
        if shap_df_global is not None:
            shap_lookup = shap_df_global.set_index('genus')['mean_abs_shap'].to_dict()
        elif fi_df is not None:
            shap_lookup = fi_df.set_index('genus')['importance'].to_dict()
        else:
            ax.set_title(f'{cancer_type}\n(No SHAP/importance data)')
            continue

        # Make genus names the row index for easy lookup
        if 'genus' in da_df.columns:
            da_df = da_df.set_index('genus')

        # Build a scatter DataFrame: one row per bacterium
        scatter_rows = []
        for genus, row in da_df.iterrows():
            shap_val = shap_lookup.get(str(genus), 0.0)  # SHAP score; 0.0 if genus not in SHAP
            scatter_rows.append({
                'genus'   : str(genus),
                'log2fc'  : float(row[fc_col]),   # fold change (x-axis)
                'adj_p'   : float(row[p_col]),    # adjusted p-value (for color)
                'shap_imp': float(shap_val)       # SHAP importance (y-axis)
            })

        scatter_df = pd.DataFrame(scatter_rows).dropna(subset=['log2fc', 'adj_p'])

        # Mark each bacterium as significant or not
        scatter_df['significant'] = scatter_df['adj_p'] < 0.05

        # Color mapping: red = significant, grey = not significant
        sig_colors = {True: '#E53935', False: '#90A4AE'}

        # Plot each group separately so the legend labels make sense
        for sig_val, group in scatter_df.groupby('significant'):
            label = 'Significant (adj_p < 0.05)' if sig_val else 'Not significant'
            ax.scatter(
                group['log2fc'],    # x: fold change
                group['shap_imp'],  # y: SHAP importance
                c=sig_colors[sig_val],
                alpha=0.6,          # semi-transparent so overlapping dots are visible
                s=40,               # dot size
                label=label,
                edgecolors='white',
                linewidths=0.3
            )

        # Annotate the top-right quadrant: enriched in cancer (FC > 0), significant, high SHAP
        # These are the most biologically interesting bacteria — important AND enriched
        top_right = scatter_df[
            (scatter_df['log2fc'] > 0) &    # enriched in cancer
            (scatter_df['significant']) &    # statistically significant
            (scatter_df['shap_imp'] > 0)     # has some SHAP importance
        ].nlargest(8, 'shap_imp')  # top 8 by SHAP score

        for _, row in top_right.iterrows():
            ax.annotate(
                row['genus'],            # label text = genus name
                xy=(row['log2fc'], row['shap_imp']),   # point to annotate
                xytext=(4, 4),           # offset text 4 points right and up
                textcoords='offset points',
                fontsize=7,
                color='#1A237E'          # dark navy blue for readability
            )

        # Reference lines to divide the plot into quadrants
        ax.axvline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.7)  # FC = 0 (vertical)
        ax.axhline(
            scatter_df['shap_imp'].quantile(0.75),  # 75th percentile SHAP (horizontal)
            color='grey', linewidth=0.8, linestyle=':', alpha=0.7, label='75th pct SHAP'
        )

        ax.set_xlabel('log2 Fold Change (Cancer / Healthy)', fontsize=12)
        ax.set_ylabel('SHAP Importance', fontsize=12)
        ax.set_title(f'{cancer_type}\nSHAP vs Differential Abundance', fontsize=13, fontweight='bold')
        ax.legend(fontsize=9, loc='upper left')
        sns.despine(ax=ax)  # remove top and right borders for a cleaner look

    plt.tight_layout()  # automatically adjust spacing between panels
    fig_path = os.path.join(FIGURES_DIR, 'fig11b_shap_da_scatter.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')  # save at high resolution
    plt.close()
    print(f'Saved: {fig_path}')

# Final status report — check which output files were created
print('\n=== Notebook 07 Complete ===')
print('Outputs generated:')
expected_outputs = [
    os.path.join(FIGURES_DIR, 'fig11_shap_beeswarm.png'),   # beeswarm (skipped if SHAP rows mismatch)
    os.path.join(FIGURES_DIR, 'fig12_shap_bar.png'),         # global importance bar chart
    os.path.join(FIGURES_DIR, 'fig11b_shap_da_scatter.png'), # SHAP vs DA scatter
    os.path.join(RESULTS_DIR, 'shap_feature_importance.csv'),# global SHAP table
    os.path.join(RESULTS_DIR, 'shap_per_class_top_genera.csv'), # per-class top 10
    os.path.join(RESULTS_DIR, 'shap_da_overlap.csv'),         # overlap analysis
]
for p in expected_outputs:
    status = 'OK' if os.path.exists(p) else 'MISSING'
    print(f'  [{status}]  {os.path.basename(p)}')